### 前置作業

In [2]:
from app import app
from database import db, POI, UserHist
import warnings
warnings.filterwarnings('ignore')

# 沒啟動 Flask server 時，程式碼開頭須把 app context 推入
app.app_context().push()
# Or 用 with 包著所有 sqlalcjemy 相關程式 e.g.
# with app.app_context():
#     db.session.commit()...

### DB key 定義

In [3]:
# db 存著兩個 table (定義於 backend/database.py)

# 1. POI
# 存 Place 資料

class POI(db.Model):
    __tablename__ = "pois"

    id = db.Column(db.Integer, primary_key=True) # place raw id
    item_id = db.Column(db.Integer, index=True) # id for model inference
    lat = db.Column(db.Float, nullable=False, index=True) # latitude
    lng = db.Column(db.Float, nullable=False, index=True) # longitude
    cat_name = db.Column(db.String(100), index=True) # category_name 

    raw_data = db.Column(db.JSON, nullable=False) # other informations but not for filtering and querying

    def to_dict(self): # db object to python dict object
        return self.raw_data

# 2. UserHist
# 存 user 造訪紀錄
class UserHist(db.Model):
    __tablename__ = "userHis"

    id = db.Column(db.Integer, primary_key=True) # unique id for every visit (for db only, seldom used)
    user_id = db.Column(db.Integer, nullable=False)
    poi_id = db.Column(db.Integer, nullable=False)
    visit_time = db.Column(db.DateTime, nullable=True)

    def to_dict(self):
        poi_record = POI.query.get(self.poi_id)
        return {
            "log_id": self.id,
            "user_id": self.user_id,
            "visit_time": self.visit_time.strftime("%Y-%m-%dT%H:%M:%SZ")
            if self.visit_time
            else None,
            "poi_detail": poi_record.to_dict() if poi_record else None,
        }


InvalidRequestError: Table 'pois' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.

### 查詢(只能用 primary key)

In [4]:
# 用 primary key 查詢 -> get
# POI : place raw id
# UserHist : db 內部 id，少用

POI.query.get(8956)

<POI 8956>

In [5]:
# use to_dict to convert db object to dict
POI.query.get(8956).to_dict()

{'raw_poi_id': 8956,
 'latitude': 32.9426550833,
 'longitude': -97.1311998333,
 'checkins_count_from_events': 40.0,
 'users_count_from_events': 28.0,
 'item_id': 2687.0,
 'spot_latitude': 32.9426550833,
 'spot_longitude': -97.1311998333,
 'category_id': 125.0,
 'category_name': 'City Hall',
 'raw_categories': [{'url': '/categories/125', 'name': 'City Hall'}],
 'photos_count': 5.0,
 'checkins_count': 335.0,
 'users_count': 146.0,
 'radius_meters': 75.0,
 'highlights_count': 0.0,
 'items_count': 10.0,
 'max_items_count': 10.0,
 'created_at': '2008-12-18T23:38:58Z'}

### 篩選

In [6]:
# 用其他 index 查詢 -> filter
[e for e in POI.query.filter((POI.cat_name == "Airport") & (POI.lat<3 ))]

[<POI 19017>,
 <POI 24136>,
 <POI 24563>,
 <POI 27904>,
 <POI 31784>,
 <POI 35044>,
 <POI 35053>,
 <POI 61281>,
 <POI 88087>,
 <POI 97081>,
 <POI 124174>,
 <POI 124380>,
 <POI 151989>,
 <POI 180439>,
 <POI 190482>,
 <POI 245444>,
 <POI 1043094>]

In [7]:
# 限制 n 筆
[e for e in POI.query.filter(POI.cat_name == "Airport", POI.lat<3 ).limit(3)]

[<POI 19017>, <POI 24136>, <POI 24563>]

In [8]:
# _.in() 找集合
target_cats = ["Airport", "Apple Store"]
[e for e in POI.query.filter(POI.cat_name.in_(target_cats)).limit(3)]

[<POI 8977>, <POI 9410>, <POI 9593>]

### 新增

In [9]:
from datetime import datetime
new_log = UserHist(user_id=111, poi_id=8956, visit_time=datetime.utcnow()) # define new log values
db.session.add(new_log) # add
db.session.commit() # sync
print(UserHist.query.filter(UserHist.user_id==111)[0].to_dict())

{'log_id': 22, 'user_id': 111, 'visit_time': '2026-04-15T15:50:56Z', 'poi_detail': {'raw_poi_id': 8956, 'latitude': 32.9426550833, 'longitude': -97.1311998333, 'checkins_count_from_events': 40.0, 'users_count_from_events': 28.0, 'item_id': 2687.0, 'spot_latitude': 32.9426550833, 'spot_longitude': -97.1311998333, 'category_id': 125.0, 'category_name': 'City Hall', 'raw_categories': [{'url': '/categories/125', 'name': 'City Hall'}], 'photos_count': 5.0, 'checkins_count': 335.0, 'users_count': 146.0, 'radius_meters': 75.0, 'highlights_count': 0.0, 'items_count': 10.0, 'max_items_count': 10.0, 'created_at': '2008-12-18T23:38:58Z'}}


### 修改

In [10]:
log_to_update = UserHist.query.get(new_log.id)
log_to_update.user_id = 888
db.session.commit()
print(UserHist.query.filter(UserHist.user_id==888)[0].to_dict())

{'log_id': 26, 'user_id': 888, 'visit_time': '2026-04-15T16:08:05Z', 'poi_detail': {'raw_poi_id': 8956, 'latitude': 32.9426550833, 'longitude': -97.1311998333, 'checkins_count_from_events': 40.0, 'users_count_from_events': 28.0, 'item_id': 2687.0, 'spot_latitude': 32.9426550833, 'spot_longitude': -97.1311998333, 'category_id': 125.0, 'category_name': 'City Hall', 'raw_categories': [{'url': '/categories/125', 'name': 'City Hall'}], 'photos_count': 5.0, 'checkins_count': 335.0, 'users_count': 146.0, 'radius_meters': 75.0, 'highlights_count': 0.0, 'items_count': 10.0, 'max_items_count': 10.0, 'created_at': '2008-12-18T23:38:58Z'}}


### 刪除

In [11]:
log_to_delete = UserHist.query.get(new_log.id)
db.session.delete(log_to_delete)
db.session.commit()
print(UserHist.query.filter(UserHist.user_id==888).count())

0
